In [1]:
%load_ext rpy2.ipython

In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
93636


In [7]:
import pandas as pd
import polars as pl

assay_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "forced-selection")
    .filter(pl.col("measurand").str.starts_with("steered_to_score:"))
    .rename({"value": "score"})
    .with_columns(
        pl.col("measurand")
        .str.replace("^steered_to_score:", "")
        .alias("target_entity_id")
    )
    .rename({
        "entity_id": "source_entity_id",
        "entity_name": "source_entity_name",
    })
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "source_entity_id",
        "source_entity_name",
        "target_entity_id",
    )
)

# Add target_entity_name from entity id/name mapping
entity_lookup = (
    df.select(
        "comparison_set_id",
        pl.col("entity_id").alias("target_entity_id"),
        pl.col("entity_name").alias("target_entity_name"),
    )
    .unique()
)

assay_df = assay_df.join(
    entity_lookup,
    on=["comparison_set_id", "target_entity_id"],
    how="left",
)

pdf = assay_df.to_pandas()

for col in [
    "model",
    "source_entity_id",
    "target_entity_id",
    "comparison_set_id",
    "assay_instance_hash",
]:
    pdf[col] = pdf[col].astype("category")

pdf["score"] = pd.to_numeric(pdf["score"])

print(pdf.dtypes)
print(pdf.head())

score                   float64
assay                       str
assay_instance_hash    category
sample_number            uint64
model                  category
comparison_set_id      category
comparison_set_name         str
source_entity_id       category
source_entity_name          str
target_entity_id       category
target_entity_name          str
dtype: object
   score             assay  assay_instance_hash  sample_number  \
0    0.0  forced-selection  3925925642981540782              0   
1    0.5  forced-selection  3925925642981540782              0   
2    0.0  forced-selection  3925925642981540782              0   
3    0.5  forced-selection  3925925642981540782              0   
4    0.0  forced-selection  3925925642981540782              0   

             model  comparison_set_id comparison_set_name source_entity_id  \
0  claude-opus-4.6  coding-assistants   coding-assistants      claude-code   
1  claude-opus-4.6  coding-assistants   coding-assistants      claude-code   
2  

In [8]:
%%R -i pdf

df <- pdf

df$model <- factor(df$model)
df$comparison_set_id <- factor(df$comparison_set_id)
df$assay_instance_hash <- factor(df$assay_instance_hash)

# Source and target should share the same entity universe
entity_levels <- sort(unique(c(
  as.character(df$source_entity_id),
  as.character(df$target_entity_id)
)))

df$source_entity_id <- factor(df$source_entity_id, levels = entity_levels)
df$target_entity_id <- factor(df$target_entity_id, levels = entity_levels)

# Sum-code only top-level direct factors
contrasts(df$model) <- contr.sum(nlevels(df$model))
contrasts(df$comparison_set_id) <- contr.sum(nlevels(df$comparison_set_id))


get_entity_levels_by_set <- function(data) {
  out <- list()

  for (g in levels(data$comparison_set_id)) {
    idx <- data$comparison_set_id == g

    out[[g]] <- sort(unique(c(
      as.character(data$source_entity_id[idx]),
      as.character(data$target_entity_id[idx])
    )))
  }

  out
}


make_local_sum_contrasts <- function(data, group_var, child_var, levels_by_group = NULL) {
  group <- data[[group_var]]
  child <- data[[child_var]]

  out <- NULL

  for (g in levels(group)) {
    idx <- group == g

    if (is.null(levels_by_group)) {
      kids <- sort(unique(as.character(child[idx])))
    } else {
      kids <- levels_by_group[[g]]
    }

    if (length(kids) <= 1) next

    C <- contr.sum(length(kids))
    rownames(C) <- kids

    M <- matrix(
      0,
      nrow = nrow(data),
      ncol = ncol(C)
    )

    M[idx, ] <- C[as.character(child[idx]), , drop = FALSE]

    colnames(M) <- paste0(
      child_var, "_within_", group_var,
      "[", g, "]_", seq_len(ncol(C))
    )

    out <- cbind(out, M)
  }

  out
}


entity_levels_by_set <- get_entity_levels_by_set(df)

Source_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "source_entity_id",
  levels_by_group = entity_levels_by_set
)

Target_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "target_entity_id",
  levels_by_group = entity_levels_by_set
)

A_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "assay_instance_hash"
)

fit <- lm(
  score ~
    model * comparison_set_id +
    Source_within_set +
    Target_within_set +
    A_within_set +
    model:Source_within_set +
    model:Target_within_set,
  data = df
)

/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "assay_instance_hash". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "sample_number". Fall back to string conversion. The error is: Cannot convert numpy array of <class 'numpy.uint64'> (R integers are signed 32-bit integers).
  warnings.warn('Error while trying to convert '


In [9]:
%%R

is_sum_coded <- function(x, name, tol = 1e-8) {
  cm <- contrasts(x)

  ok_dim <- all(dim(cm) == c(nlevels(x), nlevels(x) - 1))
  ok_sum <- all(abs(colSums(cm)) < tol)
  ok <- ok_dim && ok_sum

  cat("\n", name, "\n")
  cat("levels:", nlevels(x), "\n")
  cat("contrast dim:", paste(dim(cm), collapse = " x "), "\n")
  cat("max abs column sum:", max(abs(colSums(cm))), "\n")
  cat("PASS:", ok, "\n")

  ok
}

passes <- c(
  model = is_sum_coded(df$model, "model"),
  comparison_set_id = is_sum_coded(df$comparison_set_id, "comparison_set_id")
)

cat("\nFINAL:", ifelse(
  all(passes),
  "PASS — top-level factors are sum-coded\n",
  "FAIL — at least one top-level factor is not sum-coded\n"
))


 model 
levels: 18 
contrast dim: 18 x 17 
max abs column sum: 0 
PASS: TRUE 

 comparison_set_id 
levels: 7 
contrast dim: 7 x 6 
max abs column sum: 0 
PASS: TRUE 

FINAL: PASS — top-level factors are sum-coded


In [10]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

check_within_sum <- function(dev, group, child, label, tol = 1e-8) {
  tmp <- data.frame(
    group = as.character(group),
    child = as.character(child),
    dev = dev
  )

  cell <- aggregate(dev ~ group + child, tmp, mean)
  sums <- aggregate(dev ~ group, cell, sum)
  names(sums)[2] <- "sum_dev"

  cat("\n", label, "\n")
  print(sums)

  ok <- all(abs(sums$sum_dev) < tol)

  cat("PASS:", ok, "\n")

  ok
}

source_ok <- check_within_sum(
  term_dev(fit, "Source_within_set"),
  df$comparison_set_id,
  df$source_entity_id,
  "source/inertia deviations within comparison_set_id"
)

target_ok <- check_within_sum(
  term_dev(fit, "Target_within_set"),
  df$comparison_set_id,
  df$target_entity_id,
  "target/attraction deviations within comparison_set_id"
)

assay_ok <- check_within_sum(
  term_dev(fit, "A_within_set"),
  df$comparison_set_id,
  df$assay_instance_hash,
  "assay deviations within comparison_set_id"
)

cat("\nFINAL:", ifelse(
  source_ok && target_ok && assay_ok,
  "PASS — source, target, and assay deviations sum to zero within comparison sets\n",
  "FAIL — at least one local deviation block does not sum to zero\n"
))


 source/inertia deviations within comparison_set_id 
                   group       sum_dev
1      coding-assistants -3.279712e-18
2        email-providers -3.469447e-18
3           model-family  9.974660e-18
4 model-owner-innovation  4.336809e-18
5                   paas  8.673617e-19
6            web-browser  0.000000e+00
7       web-office-tools  0.000000e+00
PASS: TRUE 

 target/attraction deviations within comparison_set_id 
                   group       sum_dev
1      coding-assistants -6.938894e-18
2        email-providers  0.000000e+00
3           model-family -1.734723e-17
4 model-owner-innovation  1.214306e-17
5                   paas  4.336809e-19
6            web-browser  0.000000e+00
7       web-office-tools  0.000000e+00
PASS: TRUE 

 assay deviations within comparison_set_id 
                   group       sum_dev
1      coding-assistants -2.168404e-19
2        email-providers  0.000000e+00
3           model-family  8.673617e-19
4 model-owner-innovation  0.000000e+00
5

In [11]:
%%R

check_model_child_within_sum <- function(dev, model, set, child, label, tol = 1e-8) {
  tmp <- data.frame(
    model = as.character(model),
    set = as.character(set),
    child = as.character(child),
    dev = dev
  )

  cell <- aggregate(dev ~ model + set + child, tmp, mean)
  sums <- aggregate(dev ~ model + set, cell, sum)
  names(sums)[3] <- "sum_dev"

  cat("\n", label, "\n")
  cat("max abs sum:", max(abs(sums$sum_dev)), "\n")

  ok <- all(abs(sums$sum_dev) < tol)

  cat("PASS:", ok, "\n")

  if (!ok) {
    print(sums[order(-abs(sums$sum_dev)), ][1:30, ])
  } else {
    print(head(sums, 30))
  }

  ok
}

model_source_ok <- check_model_child_within_sum(
  term_dev(fit, "model:Source_within_set"),
  df$model,
  df$comparison_set_id,
  df$source_entity_id,
  "model-by-source/inertia deviations within comparison_set_id"
)

model_target_ok <- check_model_child_within_sum(
  term_dev(fit, "model:Target_within_set"),
  df$model,
  df$comparison_set_id,
  df$target_entity_id,
  "model-by-target/attraction deviations within comparison_set_id"
)

cat("\nFINAL:", ifelse(
  model_source_ok && model_target_ok,
  "PASS — model-specific source and target deviations sum to zero within each model × comparison set\n",
  "FAIL — at least one model-specific source/target deviation block does not sum to zero\n"
))


 model-by-source/inertia deviations within comparison_set_id 
max abs sum: 3.122502e-17 
PASS: TRUE 
                        model               set       sum_dev
1             claude-opus-4.6 coding-assistants -6.288373e-18
2           claude-sonnet-4.6 coding-assistants -8.673617e-19
3               deepseek-v3.2 coding-assistants -2.222614e-18
4            gemini-2.5-flash coding-assistants -6.505213e-18
5              gemini-2.5-pro coding-assistants -1.989511e-17
6                 gpt-4o-mini coding-assistants  2.818926e-18
7                     gpt-5.4 coding-assistants  3.469447e-18
8                gpt-oss-120b coding-assistants  7.264155e-18
9                      grok-4 coding-assistants -7.372575e-18
10              grok-4.1-fast coding-assistants -3.469447e-18
11     llama-3.1-70b-instruct coding-assistants  2.168404e-18
12      llama-3.1-8b-instruct coding-assistants -1.301043e-18
13               mistral-nemo coding-assistants  1.585646e-18
14         mistral-small-2603 

In [12]:
%%R

X <- model.matrix(fit)

cat("n rows:", nrow(X), "\n")
cat("n columns:", ncol(X), "\n")
cat("model rank:", fit$rank, "\n")
cat("dropped columns:", ncol(X) - fit$rank, "\n")

n rows: 50868 
n columns: 1508 
model rank: 1490 
dropped columns: 18 


In [13]:
%%R

summary(fit)


Call:
lm(formula = score ~ model * comparison_set_id + Source_within_set + 
    Target_within_set + A_within_set + model:Source_within_set + 
    model:Target_within_set, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.78222 -0.07334 -0.00569  0.05868  0.99172 

Coefficients: (18 not defined because of singularities)
                                                                                               Estimate
(Intercept)                                                                                   2.353e-01
model1                                                                                        4.881e-02
model2                                                                                        5.005e-02
model3                                                                                        1.818e-02
model4                                                                                       -4.190e-02
model5                          

In [20]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

df$.model_source_dev <- term_dev(fit, "model:Source_within_set")
df$.model_target_dev <- term_dev(fit, "model:Target_within_set")
df$.model_directed_steering_dev <- df$.model_source_dev + df$.model_target_dev
df$.fitted <- fitted(fit)

In [21]:
%%R

# I don't steer from X

source_inertia <- aggregate(
  .model_source_dev ~
    model + comparison_set_id + source_entity_id + source_entity_name,
  data = df,
  FUN = mean
)

source_inertia <- source_inertia[
  order(source_inertia$.model_source_dev),
]

head(source_inertia, 30)

                         model      comparison_set_id    source_entity_id
784              grok-4.1-fast model-owner-innovation                 xai
466                      phi-4                   paas     microsoft-azure
352              grok-4.1-fast           model-family                grok
474                gpt-4o-mini            web-browser      microsoft-edge
24                 gpt-4o-mini        email-providers             alimail
32          mistral-small-2603        email-providers             alimail
721            claude-opus-4.6            web-browser              safari
307            claude-opus-4.6       web-office-tools    google-workspace
591 nemotron-3-super-120b-a12b model-owner-innovation              nvidia
166           gemini-2.5-flash            web-browser             firefox
283               mistral-nemo            web-browser       google-chrome
351                     grok-4           model-family                grok
748              grok-4.1-fast      co

In [22]:
%%R

# I always steer to X

target_attraction <- aggregate(
  .model_target_dev ~
    model + comparison_set_id + target_entity_id + target_entity_name,
  data = df,
  FUN = mean
)

target_attraction <- target_attraction[
  order(-target_attraction$.model_target_dev),
]

head(target_attraction, 30)

                         model      comparison_set_id target_entity_id
144        qwen3.5-flash-02-23      coding-assistants           cursor
128          claude-sonnet-4.6      coding-assistants           cursor
100              grok-4.1-fast      coding-assistants      claude-code
221             gemini-2.5-pro      coding-assistants   github-copilot
658              grok-4.1-fast        email-providers      proton-mail
119     llama-3.1-70b-instruct      coding-assistants            codex
784              grok-4.1-fast model-owner-innovation              xai
136              grok-4.1-fast      coding-assistants           cursor
351                     grok-4           model-family             grok
604              grok-4.1-fast model-owner-innovation           openai
808                      phi-4        email-providers       yahoo-mail
425     llama-3.1-70b-instruct model-owner-innovation        microsoft
244              grok-4.1-fast        email-providers            gmail
233   